In [1]:
# !pip install ipython==8.1.0

In [15]:
# # Comment the following if you are running your code locally
# # This mounts your Google Drive to the Colab VM.
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder = '/content/drive/MyDrive'
# notebook_folder = drive_folder + '/project'
# %cd {notebook_folder}

# You don't need to run unless the csv files are missing
# %cd ./cs682/data
# !bash prepare.sh
# %cd ../../

In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from transformers import get_linear_schedule_with_warmup
import json

from cs682.models.teacher import BERTTeacher
from cs682.models.student import FunnelTransformer
from cs682.data.loader import IMDBDataset, YelpDataset, AmazonDataset
from cs682.evaluator import evaluate
from tqdm import tqdm

In [25]:
class StudentTrainConfig:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __repr__(self):
        return f"StudentTrainConfig({self.__dict__})"


class DistillationDataset(torch.utils.data.Dataset):
    """
    teacher_signals[i] must correspond to base_dataset[i], so the teacher
    must have been run with shuffle=False over the same dataset ordering.
    """

    def __init__(self, base_dataset, teacher_signals):
        assert len(base_dataset) == len(teacher_signals), (
            f"Dataset size {len(base_dataset)} != teacher signals size {len(teacher_signals)}"
        )
        self.base = base_dataset
        self.signals = teacher_signals

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        text, label = self.base[idx]
        sig = self.signals[idx]
        return text, label, sig["teacher_logits"], sig["teacher_cls_hiddens"]


def make_distill_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        texts, labels, t_logits, t_cls_per_sample = zip(*batch)
        enc = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        enc["teacher_logits"] = torch.stack(list(t_logits))  # (B, C)
        # t_cls_per_sample: B items each a list of K tensors (D,)
        # transpose -> list of K tensors each (B, D)
        K = len(t_cls_per_sample[0])
        enc["teacher_cls_hiddens"] = [
            torch.stack([t_cls_per_sample[b][k] for b in range(len(batch))])
            for k in range(K)
        ]
        return enc

    return collate_fn


def make_collate_fn(tokenizer, max_length):
    def collate_fn(batch):
        texts, labels = zip(*batch)
        enc = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc

    return collate_fn


def check_accuracy(
    model: FunnelTransformer, loader: DataLoader, device, max_batches: int = None
):
    model.eval()
    correct = 0
    total = 0
    use_amp = device.type == "cuda"
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["labels"].to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
            predicted = torch.argmax(out["logits"], dim=1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)
    return correct / total


def run_teacher_model(
    teacher_model: BERTTeacher, train_loader: DataLoader, num_classes, device
):
    print("Running Teacher model to get signals.")
    teacher_model.eval()

    # need logits, and CLS hiddens
    signals = []
    with torch.no_grad():
        for i, batch in tqdm(enumerate(train_loader), "Getting teacher signals"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            # (B, C), (B, D)
            out = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            # add to signals per each item
            for j in range(input_ids.size(0)):
                signals.append(
                    {
                        "teacher_logits": out["logits"][j].cpu(),
                        "teacher_cls_hiddens": [h[j].cpu() for h in out["cls_hiddens"]],
                    }
                )

    return signals


def distillation_loss(
    student_out: dict,
    teacher_logits: torch.Tensor,
    teacher_cls: list[torch.Tensor],
    labels: torch.Tensor,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
    temperature: float = 4.0,
) -> dict[str, torch.Tensor]:
    """
    Combined distillation objective:
        L = alpha * L_task  +  beta * L_logit  +  gamma * L_layer
    """
    s_logits = student_out["logits"]
    s_cls_list = student_out["cls_hiddens"]

    l_task = F.cross_entropy(s_logits, labels)

    s_soft = F.log_softmax(s_logits / temperature, dim=-1)
    t_soft = F.softmax(teacher_logits / temperature, dim=-1)
    l_logit = F.kl_div(s_soft, t_soft, reduction="batchmean") * (temperature**2)

    if len(s_cls_list) != len(teacher_cls):
        raise ValueError(
            f"Expected {len(s_cls_list)} teacher CLS tensors "
            f"(one per block), got {len(teacher_cls)}"
        )
    l_layer = sum(
        F.mse_loss(s_cls, t_cls) for s_cls, t_cls in zip(s_cls_list, teacher_cls)
    ) / len(s_cls_list)

    loss = alpha * l_task + beta * l_logit + gamma * l_layer

    return {
        "loss": loss,
        "l_task": l_task.detach(),
        "l_logit": l_logit.detach(),
        "l_layer": l_layer.detach(),
    }


def train(
    config: StudentTrainConfig, train_loader: DataLoader, validation_loader: DataLoader
):
    print(config)

    optimizer = config.optimizer
    scheduler = config.scheduler
    student_model = config.student_model
    max_grad_norm = config.max_grad_norm

    epochs = config.epochs
    log_every_k = config.log_every_k
    accuracy_every_k = config.accuracy_every_k
    alpha = config.alpha
    beta = config.beta
    gamma = config.gamma
    temperature = config.temperature

    device = config.device
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)

    student_model.to(device)

    losses = []
    train_accuracies = []
    validation_accuracies = []

    for epoch in range(epochs):
        student_model.train()
        for i, batch in enumerate(
            tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")
        ):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            teacher_logits = batch["teacher_logits"].to(device)
            teacher_cls = [t.to(device) for t in batch["teacher_cls_hiddens"]]

            optimizer.zero_grad()

            with torch.autocast(device_type=device.type, enabled=use_amp):
                student_out = student_model(
                    input_ids=input_ids, attention_mask=attention_mask
                )
                # Align student CLS hiddens to the number of teacher mapped layers:
                # teacher_cls has K items (one per mapped layer); take the last K student blocks.
                n_t = len(teacher_cls)
                aligned_out = {
                    **student_out,
                    "cls_hiddens": student_out["cls_hiddens"][-n_t:],
                }
                loss_dict = distillation_loss(
                    student_out=aligned_out,
                    teacher_logits=teacher_logits,
                    teacher_cls=teacher_cls,
                    labels=labels,
                    alpha=alpha,
                    beta=beta,
                    gamma=gamma,
                    temperature=temperature,
                )

            scaler.scale(loss_dict["loss"]).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            losses.append(loss_dict["loss"].item())

            if i % log_every_k == 0:
                print(
                    f"Epoch {epoch + 1} | Step {i} | "
                    f"loss={loss_dict['loss']:.4f} | "
                    f"l_task={loss_dict['l_task']:.4f} | "
                    f"l_logit={loss_dict['l_logit']:.4f} | "
                    f"l_layer={loss_dict['l_layer']:.4f}"
                )

            if i % accuracy_every_k == 0:
                train_acc = check_accuracy(
                    student_model, train_loader, device, max_batches=20
                )
                val_acc = check_accuracy(student_model, validation_loader, device)
                train_accuracies.append(train_acc)
                validation_accuracies.append(val_acc)
                print(f"  Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")
                student_model.train()

    return {
        "losses": losses,
        "train_accuracies": train_accuracies,
        "validation_accuracies": validation_accuracies,
    }

In [29]:
dataset_name = "imdb" # imdb, yelp, amazon
is_two_classes = True # only applied for Amazon or Yelp, leave as True for IMDB
epochs = 3
learning_rate = 2e-5
weight_decay = 1e-4
warmup_ratio = 0.06
max_grad_norm = 1.0
validation_pct = 0.2
batch_size = 32
num_workers = 4
log_every_k = 50
accuracy_every_k = 200
max_length = 128
dropout = 0.1
teacher_model_save_path = f"./models/teacher_{dataset_name}_{is_two_classes}.pt"

student_model_save_path = f"./models/student_{dataset_name}_{is_two_classes}.pt"
student_train_save_path = f"./models/student_{dataset_name}_{is_two_classes}.json"
mapped_layer_indices = [4, 8] # you can leave this as is, during distillation it will change

# change from args to regular variables
num_classes = 2 if is_two_classes else 5

if dataset_name == "imdb":
    dataset = IMDBDataset()
    num_classes = 2
elif dataset_name == "yelp":
    dataset = YelpDataset(is_two_classes=is_two_classes)
elif dataset_name == "amazon":
    dataset = AmazonDataset(is_two_classes=is_two_classes)
else:
    raise ValueError()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Handle Teacher model before training
teacher_model, teacher_tokenizer = BERTTeacher.from_pretrained(
    num_classes=num_classes,
    mapped_layer_indices=mapped_layer_indices,
    dropout=dropout,
)

teacher_checkpoint = torch.load(teacher_model_save_path, map_location=device)
teacher_model.load_state_dict(teacher_checkpoint["model_state_dict"])
teacher_model.to(device)

collate_fn = make_collate_fn(teacher_tokenizer, max_length)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=num_workers, pin_memory=True)

teacher_signals = run_teacher_model(teacher_model, train_loader, num_classes, device)
del teacher_model, teacher_tokenizer

   label                                               text
0      1  Bromwell High is a cartoon comedy. It ran at t...
1      1  Homelessness (or Houselessness as George Carli...
2      1  Brilliant over-acting by Lesley Ann Warren. Be...
3      1  This is easily the most underrated film inn th...
4      1  This is not the typical Mel Brooks film. It wa...
Teacher layers: 8  |  Student blocks: 2Student block  ->  Teacher layer (hidden_states index)
    Block 1  ->  Layer 4
    Block 2  ->  Layer 8
Running Teacher model to get signals.


Getting teacher signals: 782it [00:15, 51.01it/s]


In [26]:
student_model, student_tokenizer = FunnelTransformer.from_bert(
    block_layers=[2, 2, 2],
    num_classes=num_classes,
    dropout=dropout,
)

n = len(dataset)
val_size = int(n * validation_pct)
train_size = n - val_size
perm = torch.randperm(n).tolist()
train_indices, val_indices = perm[:train_size], perm[train_size:]

train_signals = [teacher_signals[i] for i in train_indices]
train_base    = Subset(dataset, train_indices)
val_subset    = Subset(dataset, val_indices)

distill_dataset = DistillationDataset(train_base, train_signals)

distill_collate_fn = make_distill_collate_fn(student_tokenizer, max_length)
val_collate_fn     = make_collate_fn(student_tokenizer, max_length)

distill_loader = DataLoader(
    distill_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=distill_collate_fn,
    num_workers=num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=val_collate_fn,
    num_workers=num_workers,
    pin_memory=True,
)

optimizer    = torch.optim.AdamW(student_model.parameters(), lr=learning_rate, weight_decay=weight_decay)
total_steps  = len(distill_loader) * epochs
warmup_steps = int(total_steps * warmup_ratio)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

config = StudentTrainConfig(
    student_model=student_model,
    optimizer=optimizer,
    scheduler=scheduler,
    max_grad_norm=max_grad_norm,
    epochs=epochs,
    log_every_k=log_every_k,
    accuracy_every_k=accuracy_every_k,
    device=device,
    alpha=1.0,
    beta=1.0,
    gamma=1.0,
    temperature=4.0,
)

history = train(config, distill_loader, val_loader)

Loading 'google/bert_uncased_L-8_H-512_A-8' for embedding initialisation ...
  Copied token embeddings  (30522 x 512)
  Copied positional embeddings (first 512 positions)
  Block config: [2, 2, 2]  |  num_classes: 2
StudentTrainConfig({'student_model': FunnelTransformer(
  (token_emb): Embedding(30522, 512, padding_idx=0)
  (pos_emb): Embedding(512, 512)
  (emb_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (emb_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-2): 3 x FunnelBlock(
      (layers): ModuleList(
        (0-1): 2 x TransformerLayer(
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (ffn): Sequential(
            (0): Linear(in_features=512, out_features=2048, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.1, inplace=False)
            (3): Linear(in_features=2048, out_features=512, bias=True)
  

Epoch 1/3:   0%|          | 0/625 [00:00<?, ?it/s]

Epoch 1 | Step 0 | loss=5.2294 | l_task=0.7354 | l_logit=3.1354 | l_layer=1.3586


Epoch 1/3:   1%|          | 6/625 [00:04<05:44,  1.80it/s]

  Train acc: 0.4969 | Val acc: 0.4950


...

Epoch 3/3:  96%|█████████▌| 598/625 [00:19<00:00, 47.87it/s]

Epoch 3 | Step 600 | loss=1.3363 | l_task=0.4909 | l_logit=0.4591 | l_layer=0.3863


Epoch 3/3:  97%|█████████▋| 608/625 [00:22<00:01,  8.57it/s]

  Train acc: 0.8656 | Val acc: 0.8210


Epoch 3/3: 100%|██████████| 625/625 [00:22<00:00, 27.56it/s]


In [30]:
num_classes = 2 if is_two_classes else 5
if dataset_name == "imdb":
    dataset = IMDBDataset(split="test")
    num_classes = 2
elif dataset_name == "yelp":
    dataset = YelpDataset(split="test", is_two_classes=is_two_classes)
elif dataset_name == "amazon":
    dataset = AmazonDataset(split="test", is_two_classes=is_two_classes)
else:
    raise ValueError()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

evaluate(student_model, loader, num_classes, device)

   label                                               text
0      1  I went and saw this movie last night after bei...
1      1  Actor turned director Bill Paxton follows up h...
2      1  As a recreational golfer with some knowledge o...
3      1  I saw this film in a sneak preview, and it is ...
4      1  Bill Paxton has taken the true story of the 19...
Starting evaluation.
0 / 782 Complete
50 / 782 Complete
100 / 782 Complete
150 / 782 Complete
200 / 782 Complete
250 / 782 Complete
300 / 782 Complete
350 / 782 Complete
400 / 782 Complete
450 / 782 Complete
500 / 782 Complete
550 / 782 Complete
600 / 782 Complete
650 / 782 Complete
700 / 782 Complete
750 / 782 Complete

Overall Accuracy:           0.8214
Overall Classification Error: 0.1786


Class         Precision       Recall
------------------------------------
0                0.8178       0.8270
1                0.8251       0.8157


(0.82136, 0.17864000000000002)